In [ ]:
import pandas as pd
from pathlib import Path
import os
import random
import numpy as np
import cv2
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import keras
from torchvision.models.video import mvit_v2_s , MViT_V2_S_Weights
import torch.nn.functional as F
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score , roc_auc_score
from pathlib import Path
import math
from sklearn.metrics import  classification_report, confusion_matrix




c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

os.environ["KERAS_BACKEND"] = "torch"

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



Keras version: 3.15.1
Keras backend: tensorflow
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
df = pd.read_csv('nexar_data/metadata.csv')

In [4]:
# target column 
df["label"] = df["time_of_event"].notna().astype("int64")

In [5]:

log_path = "nexar_data/processed_frames_matched2/clean_balanced_log.csv"
data = pd.read_csv(log_path)

data = data[data["status"].isin(["ok", "ok_existing"])].copy()
data = data.dropna(subset=["clip_path", "label"])
data["label"] = data["label"].astype(int)

train_df, temp_df = train_test_split(data , test_size=0.30 , random_state=42 , stratify=data["label"])
val_df, test_df = train_test_split(temp_df , test_size=0.50 , random_state=42 , stratify=temp_df["label"])

train_df.to_csv("nexar_data/train.csv", index=False)
val_df.to_csv("nexar_data/val.csv", index=False)
test_df.to_csv("nexar_data/test.csv", index=False)

for name, part in [("Train", train_df) , ("Validation", val_df) , ("Test", test_df)]:
    print(f"\n{name}: {len(part)} videos")
    print(part["label"].value_counts().sort_index())


Train: 789 videos
label
0    394
1    395
Name: count, dtype: int64

Validation: 169 videos
label
0    85
1    84
Name: count, dtype: int64

Test: 170 videos
label
0    85
1    85
Name: count, dtype: int64


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(44)
 
class NexarDatasetAug(Dataset):
 
    def __init__(self, dataframe, train=False, use_flip=True):
        self.data = dataframe.reset_index(drop=True)
        self.train = train
        self.use_flip = use_flip
        self.mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
        self.std = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)
 
    def __len__(self):
        return len(self.data)
 
    def _augment(self, x):  
 
        C, T, H, W = x.shape

        # random jitter 
        if random.random() < 0.3:
            drop = random.randint(1, 2)
            keep = torch.randperm(T - drop)[:T - drop].sort().values
            x = x[:, keep]
            while x.shape[1] < T:
                x = torch.cat([x, x[:, -1:]], dim=1)

 
        # random resized 
        area = random.uniform(0.6, 1.0) * H * W
        ratio = random.uniform(3 / 4, 4 / 3)
        w = min(W, int(round(math.sqrt(area * ratio))))
        h = min(H, int(round(math.sqrt(area / ratio))))
        top = random.randint(0, H - h)
        left = random.randint(0, W - w)
        x = x[:, :, top:top + h, left:left + w]
        x = F.interpolate(x.permute(1, 0, 2, 3), size=(H, W),
                          mode="bilinear", align_corners=False).permute(1, 0, 2, 3)
 
 
        # brightness / contrast jitter
        x = x * random.uniform(0.8, 1.2)
        m = x.mean()
        x = (x - m) * random.uniform(0.8, 1.2) + m
        x = x.clamp(0.0, 1.0)

 
        # horizontal flip 
        if self.use_flip and random.random() < 0.5:
            x = x.flip(-1)
        return x

 
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        frames = torch.from_numpy(np.load(row["clip_path"]).copy())
        frames = frames.permute(3, 0, 1, 2).float() / 255.0  # THWC -> CTHW
 
        if self.train:
            frames = self._augment(frames)
 
        frames = (frames - self.mean) / self.std
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return frames, label
 
 

def set_backbone_trainable(model, trainable):
    for name, p in model.backbone.named_parameters():
        if not name.startswith("head"):
            p.requires_grad = trainable
 
 
def build_optimizer(model, lr_backbone=1e-5, lr_head=1e-4, weight_decay=0.05):
    head_params = list(model.backbone.head.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return AdamW(
        [
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ],
        weight_decay=weight_decay,
    )


# makiing data sets 
train_dataset = NexarDatasetAug(train_df, train=True)
val_dataset   = NexarDatasetAug(val_df,   train=False)
test_dataset  = NexarDatasetAug(test_df,  train=False)


# loading them on gpu / chosing batch size here 
train_loader = DataLoader(train_dataset , batch_size=4 , shuffle=True , num_workers=0 , pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset , batch_size=4 , shuffle=False , num_workers=0 , pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset , batch_size=4 , shuffle=False , num_workers=0 , pin_memory=torch.cuda.is_available())


videos, labels = next(iter(train_loader))
print("Video batch:", videos.shape)
print("Labels:", labels)


Video batch: torch.Size([4, 3, 16, 224, 224])
Labels: tensor([1, 0, 0, 1])
Device available: True


In [ ]:


# model architecture 
class NexarCollisionModel(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        weights = MViT_V2_S_Weights.DEFAULT if pretrained else None
        self.backbone = mvit_v2_s(weights=weights)
        in_features = self.backbone.head[1].in_features
        self.backbone.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)



model = NexarCollisionModel(num_classes=2, pretrained=True).to(device)

print(model)

NexarCollisionModel(
  (backbone): MViT(
    (conv_proj): Conv3d(3, 96, kernel_size=(3, 7, 7), stride=(2, 4, 4), padding=(1, 3, 3))
    (pos_encoding): PositionalEncoding()
    (blocks): ModuleList(
      (0): MultiscaleBlock(
        (norm1): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (attn): MultiscaleAttention(
          (qkv): Linear(in_features=96, out_features=288, bias=True)
          (project): Sequential(
            (0): Linear(in_features=96, out_features=96, bias=True)
          )
          (pool_q): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=96, bias=False)
            (norm_act): Sequential(
              (0): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
            )
          )
          (pool_k): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 8, 8), p

In [ ]:

# loss function / optimizer / schedule and scaler 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
val_criterion = nn.CrossEntropyLoss()   
optimizer = build_optimizer(model  , lr_backbone=1e-5, lr_head=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer , T_max=15 , eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda" , enabled=(device.type == "cuda"))
checkpoint_path = Path("nexar_data/best_mvit_collision.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# initializing constant 
num_epochs = 15
patience = 3
UNFREEZE_EPOCH = 3
best_val_auc = 0.0
epochs_without_improvement = 0
history = {
    "train_loss": [], 
    "val_loss": [],
    "train_f1": [], 
    "val_f1": [],
    "train_acc": [], 
    "val_acc": [],
    "val_auc": [],                                
}



# stopping backbone trainable to ensure model wont overfitt early 
set_backbone_trainable(model, False)  
# running model on training data and validation data 
for epoch in range(num_epochs):
 
    if epoch == UNFREEZE_EPOCH:
        # unpausing it after 3 epoch 
        set_backbone_trainable(model, True)
        print("    Backbone unfrozen")
 
    # training
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    train_preds = []
    train_targets = []
 
    for videos, labels in train_loader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
 
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(videos)
            loss = criterion(outputs, labels)
 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
 
        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_count += batch_size
 
        predictions = outputs.argmax(dim=1)
 
        train_preds.extend(predictions.detach().cpu().tolist())
        train_targets.extend(labels.cpu().tolist())
 
    train_loss = train_loss_sum / train_count
    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro", zero_division=0)
 
    # validation
    model.eval()
 
    val_loss_sum = 0.0
    val_count = 0
    val_preds = []
    val_targets = []
    val_probs = []                           
 
    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
 
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(videos)
                loss = val_criterion(outputs, labels)     
 
            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_count += batch_size
 
            predictions = outputs.argmax(dim=1)
 
            val_preds.extend(predictions.cpu().tolist())
            val_targets.extend(labels.cpu().tolist())
            val_probs.extend(torch.softmax(outputs.float(), dim=1)[:, 1].cpu().tolist())  
 
    val_loss = val_loss_sum / val_count
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    val_precision = precision_score(val_targets, val_preds, average="macro", zero_division=0)
    val_recall = recall_score(val_targets, val_preds, average="macro", zero_division=0)
    val_auc = roc_auc_score(val_targets, val_probs)                                         
 
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
 
    # updating learning rate
    scheduler.step()


    # logggggs 
    print(
        f"Epoch [{epoch + 1:02d}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Train Macro F1: {train_f1:.4f} | "
        f"Val Macro F1: {val_f1:.4f} | "
        f"Val AUC: {val_auc:.4f}")
 
    print(
        f"    Val Macro Precision: {val_precision:.4f} | "
        f"Val Macro Recall: {val_recall:.4f} | "
        f"LR backbone: {optimizer.param_groups[0]['lr']:.2e} | "
        f"LR head: {optimizer.param_groups[1]['lr']:.2e}"
    )
 
    # saving best model
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_auc": best_val_auc,
                "val_loss": val_loss,
                "val_f1": val_f1,
                "class_names": ["no_collision", "collision"]
            },
            checkpoint_path
        )

        print(f"    Saved best model: {checkpoint_path}")

    elif epoch >= UNFREEZE_EPOCH:
        epochs_without_improvement += 1
 
    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break
 
best = torch.load(checkpoint_path, map_location=device)


# loggggs 
print(f"\nBest val AUC: {best['best_val_auc']:.4f} (epoch {best['epoch']}) | "
      f"val loss: {best['val_loss']:.4f} | val F1: {best['val_f1']:.4f}")
print(f"Checkpoint saved at: {checkpoint_path}")


Epoch [01/15] Train Loss: 0.6795 | Val Loss: 0.6466 | Train Acc: 0.5665 | Val Acc: 0.6272 | Train Macro F1: 0.5659 | Val Macro F1: 0.5945 | Val AUC: 0.7377
    Val Macro Precision: 0.6836 | Val Macro Recall: 0.6256 | LR backbone: 9.89e-06 | LR head: 9.89e-05
    Saved best model: nexar_data\best_mvit_collision.pth
Epoch [02/15] Train Loss: 0.6474 | Val Loss: 0.7111 | Train Acc: 0.6350 | Val Acc: 0.5621 | Train Macro F1: 0.6350 | Val Macro F1: 0.4683 | Val AUC: 0.7513
    Val Macro Precision: 0.6963 | Val Macro Recall: 0.5597 | LR backbone: 9.57e-06 | LR head: 9.57e-05
    Saved best model: nexar_data\best_mvit_collision.pth
Epoch [03/15] Train Loss: 0.6395 | Val Loss: 0.6153 | Train Acc: 0.6515 | Val Acc: 0.6509 | Train Macro F1: 0.6511 | Val Macro F1: 0.6379 | Val AUC: 0.7556
    Val Macro Precision: 0.6784 | Val Macro Recall: 0.6520 | LR backbone: 9.05e-06 | LR head: 9.05e-05
    Saved best model: nexar_data\best_mvit_collision.pth
    Backbone unfrozen
Epoch [04/15] Train Loss: 0.62

In [ ]:



checkpoint = torch.load("nexar_data/best_mvit_collision.pth", map_location=device , weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

probs = []
labels = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(x)

        batch_probs = torch.softmax(
            logits.float(), dim=1
        )[:, 1]

        probs.extend(batch_probs.cpu().numpy())
        labels.extend(y.cpu().numpy())

probs = np.array(probs)
labels = np.array(labels)


threshold = 0.50

preds = (probs >= threshold).astype(int)




##### logggggggs
print("=" * 50)
print("TEST RESULTS")
print("=" * 50)

print(f"Samples         : {len(labels)}")
print(f"Positive        : {(labels == 1).sum()}")
print(f"Negative        : {(labels == 0).sum()}")

print(f"Threshold       : {threshold:.2f}")
print(f"Accuracy        : {accuracy_score(labels, preds):.4f}")
print(f"Macro F1        : {f1_score(labels, preds, average='macro'):.4f}")
print(f"Macro Precision : {precision_score(labels, preds, average='macro', zero_division=0):.4f}")
print(f"Macro Recall    : {recall_score(labels, preds, average='macro', zero_division=0):.4f}")
print(f"ROC-AUC         : {roc_auc_score(labels, probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(labels, preds))

print("\nClassification Report:")
print(classification_report(labels, preds, digits=4))



print("\nTHRESHOLD COMPARISON")
print("-" * 65)

for threshold in np.arange(0.25, 0.76, 0.05):
    threshold_preds = (probs >= threshold).astype(int)
    macro_f1 = f1_score(labels, threshold_preds, average="macro")
    positive_recall = recall_score(labels, threshold_preds, pos_label=1, zero_division=0)
    positive_precision = precision_score(labels, threshold_preds, pos_label=1, zero_division=0)
    
    print(
        f"Threshold: {threshold:.2f} | "
        f"Macro F1: {macro_f1:.4f} | "
        f"Collision Precision: {positive_precision:.4f} | "
        f"Collision Recall: {positive_recall:.4f}"
    )


results = pd.DataFrame({"video_id": test_df["video_id"].values, "label": labels, "collision_probability": probs, "predicted_label": preds})
results.to_csv("nexar_data/test_predictions.csv", index=False)
print("\nSaved: nexar_data/test_predictions.csv")


TEST RESULTS
Samples         : 170
Positive        : 85
Negative        : 85
Threshold       : 0.50
Accuracy        : 0.7118
Macro F1        : 0.7095
Macro Precision : 0.7186
Macro Recall    : 0.7118
ROC-AUC         : 0.7715

Confusion Matrix:
[[68 17]
 [32 53]]

Classification Report:
              precision    recall  f1-score   support

           0     0.6800    0.8000    0.7351        85
           1     0.7571    0.6235    0.6839        85

    accuracy                         0.7118       170
   macro avg     0.7186    0.7118    0.7095       170
weighted avg     0.7186    0.7118    0.7095       170


THRESHOLD COMPARISON
-----------------------------------------------------------------
Threshold: 0.25 | Macro F1: 0.4735 | Collision Precision: 0.5380 | Collision Recall: 1.0000
Threshold: 0.30 | Macro F1: 0.5765 | Collision Precision: 0.5725 | Collision Recall: 0.9294
Threshold: 0.35 | Macro F1: 0.5991 | Collision Precision: 0.5827 | Collision Recall: 0.8706
Threshold: 0.40 | Macr

In [11]:
rng = np.random.default_rng(42)
aucs, f1s = [], []
for _ in range(1000):
    idx = rng.integers(0, len(labels), len(labels))
    if len(np.unique(labels[idx])) < 2: continue
    aucs.append(roc_auc_score(labels[idx], probs[idx]))
    f1s.append(f1_score(labels[idx], preds[idx], average="macro"))

print(f"Test AUC : {roc_auc_score(labels, probs):.4f} "
      f"[95% CI {np.percentile(aucs,2.5):.4f}, {np.percentile(aucs,97.5):.4f}]")
print(f"Test F1  : {f1_score(labels, preds, average='macro'):.4f} "
      f"[95% CI {np.percentile(f1s,2.5):.4f}, {np.percentile(f1s,97.5):.4f}]")

Test AUC : 0.7715 [95% CI 0.6916, 0.8440]
Test F1  : 0.7095 [95% CI 0.6335, 0.7767]


In [ ]:


#### test section for checking data leakage / not part of the model / let them be as it 
def static_frame(x):  
    return x[:, :, :1].expand(-1, -1, x.shape[2], -1, -1).contiguous()

def borders_only(x): 
    m = torch.ones_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 0
    return x * m

def center_only(x): 
    m = torch.zeros_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 1
    return x * m

def shuffle_time(x):  
    return x[:, :, torch.randperm(x.shape[2])]

@torch.no_grad()
def macro_f1(model, loader, device, transform=lambda x: x):
    model.eval()
    preds, targets = [], []
    for videos, labels in loader:
        out = model(transform(videos).to(device))
        preds += out.argmax(1).cpu().tolist()
        targets += labels.tolist()
    return f1_score(targets, preds, average="macro", zero_division=0)

def part_c(model, val_loader, device):
    variants = {
        "normal": lambda x: x,
        "static frame": static_frame,
        "shuffled time": shuffle_time,
        "borders only": borders_only,
        "center only": center_only,
    }
    for name, fn in variants.items():
        print(f"{name:14s} val macro-F1 = {macro_f1(model, val_loader, device, fn):.3f}")

In [12]:
ckpt = torch.load("nexar_data/best_mvit_collision.pth", map_location=device)
model = NexarCollisionModel(num_classes=2, pretrained=False).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_loader = DataLoader(NexarDatasetAug(val_df), batch_size=4, shuffle=False, num_workers=0)
part_c(model, val_loader, device)

normal         val macro-F1 = 0.707
static frame   val macro-F1 = 0.488
shuffled time  val macro-F1 = 0.667
borders only   val macro-F1 = 0.678
center only    val macro-F1 = 0.706


In [ ]:


def keep_strip(top=0.0, bottom=0.0, sides=0.0):
    def fn(x):  
        H, W = x.shape[-2:]
        m = torch.zeros_like(x)
        if top:
            m[..., : int(H * top), :] = 1
        if bottom:
            m[..., H - int(H * bottom):, :] = 1
        if sides:
            m[..., :, : int(W * sides)] = 1
            m[..., :, W - int(W * sides):] = 1
        return x * m
    return fn

@torch.no_grad()
def predict(model, loader, device, transform=lambda x: x):
    model.eval()
    probs, labels = [], []
    for videos, y in loader:
        logits = model(transform(videos).to(device))
        probs += torch.softmax(logits, dim=1)[:, 1].cpu().tolist()
        labels += y.tolist()
    return np.array(probs), np.array(labels)

def strip_report(model, loader, device):
    variants = {
        "normal": lambda x: x,
        "bottom 20% only (hood/dashboard)": keep_strip(bottom=0.2),
        "top 20% only (sky/buildings)": keep_strip(top=0.2),
        "left+right 20% only": keep_strip(sides=0.2),
    }
    for name, fn in variants.items():
        p, y = predict(model, loader, device, fn)
        f1 = f1_score(y, (p > 0.5).astype(int), average="macro", zero_division=0)
        print(f"{name:36s} F1 = {f1:.3f}   AUC = {roc_auc_score(y, p):.3f}   "
              f"predicted-positive rate = {(p > 0.5).mean():.2f}")


def stratified_report(probs, labels, val_df, meta_csv="nexar_data/metadata.csv"):
    meta = pd.read_csv(meta_csv)[["video_id", "light_conditions", "weather", "scene"]]
    d = val_df.reset_index(drop=True)[["video_id", "label"]].merge(meta, on="video_id", how="left")
    assert len(d) == len(probs), "val_df and loader order/length do not match"
    assert (d["label"].to_numpy() == labels).all(), "labels do not line up with val_df"

    d["prob"] = probs
    d["pred"] = (probs > 0.5).astype(int)

    print(f"Overall: F1 = {f1_score(d.label, d.pred, average='macro'):.3f}   "
          f"AUC = {roc_auc_score(d.label, d.prob):.3f}\n")

    for col in ["light_conditions", "scene", "weather"]:
        print(f"-- by {col} --")
        for g, s in d.groupby(col):
            if len(s) < 20 or s.label.nunique() < 2:
                print(f"  {g}: n={len(s)} (too small / one class, skipped)")
                continue
            print(f"  {g}: n={len(s)}, positive share={s.label.mean():.2f}, "
                  f"F1={f1_score(s.label, s.pred, average='macro'):.3f}, "
                  f"AUC={roc_auc_score(s.label, s.prob):.3f}")
        print()

In [13]:
strip_report(model, val_loader, device)
probs, labels = predict(model, val_loader, device)
stratified_report(probs, labels, val_df)

normal                               F1 = 0.745   AUC = 0.792   predicted-positive rate = 0.43
bottom 20% only (hood/dashboard)     F1 = 0.368   AUC = 0.569   predicted-positive rate = 0.97
top 20% only (sky/buildings)         F1 = 0.512   AUC = 0.592   predicted-positive rate = 0.21
left+right 20% only                  F1 = 0.622   AUC = 0.674   predicted-positive rate = 0.39
Overall: F1 = 0.745   AUC = 0.792

-- by light_conditions --
  Bright: n=1 (too small / one class, skipped)
  Dark: n=6 (too small / one class, skipped)
  Normal: n=207, positive share=0.49, F1=0.742, AUC=0.793
  Twilight: n=11 (too small / one class, skipped)

-- by scene --
  Highway: n=48, positive share=0.42, F1=0.808, AUC=0.879
  Industrial: n=2 (too small / one class, skipped)
  Other: n=7 (too small / one class, skipped)
  Rural: n=5 (too small / one class, skipped)
  Sub-urban: n=46, positive share=0.33, F1=0.712, AUC=0.757
  Urban: n=117, positive share=0.62, F1=0.698, AUC=0.759

-- by weather --
  Clear